# PDM Training Pipeline v3

**Purpose**: Generate training data, engineer features, train models, register to Snowflake Model Registry

## Specifications:
- **Pumps**: 120 (60 normal, 60 with failures)
- **Duration**: 45 days at 5-minute intervals = 12,960 rows/pump = 1.56M total rows
- **Failure Windows** (longer for gradual degradation):
  - BEARING_WEAR: 35-50 days
  - SEAL_LEAK: 25-40 days  
  - VALVE_FAILURE: 14-25 days
  - OVERHEATING: 7-14 days
  - CAVITATION: 7-14 days
- **Labels**: Early 25% of window = NORMAL, rest = failure mode
- **RUL**: Countdown from failure day (with variance)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
session.sql("USE DATABASE PDM_DEMO").collect()
session.sql("USE SCHEMA ML").collect()
print("Connected to PDM_DEMO.ML")

In [ ]:
# Configuration
TRAIN_PUMPS = 120
TRAIN_DAYS = 45
INTERVAL_MINUTES = 5
ROWS_PER_DAY = 24 * 60 // INTERVAL_MINUTES  # 288

# Longer failure windows for gradual degradation
FAILURE_MODES = {
    'BEARING_WEAR': (35, 50),
    'SEAL_LEAK': (25, 40),
    'VALVE_FAILURE': (14, 25),
    'OVERHEATING': (7, 14),
    'CAVITATION': (7, 14)
}

NOMINAL_RANGES = {
    'suction_pressure': (20, 80),
    'discharge_pressure': (80, 250),
    'flow_rate': (100, 1000),
    'motor_current': (50, 300),
    'pump_speed': (1500, 3600),
    'bearing_temp': (120, 180),
    'casing_temp': (100, 160),
    'vibration_rms': (0.1, 0.5),
    'valve_position': (10, 90),
    'leak_rate': (0, 0.5)
}

LABELS = ['NORMAL', 'BEARING_WEAR', 'CAVITATION', 'VALVE_FAILURE', 'OVERHEATING', 'SEAL_LEAK', 'OFFLINE']
print(f"Config: {TRAIN_PUMPS} pumps × {TRAIN_DAYS} days × {ROWS_PER_DAY} rows/day = {TRAIN_PUMPS * TRAIN_DAYS * ROWS_PER_DAY:,} total rows")

In [ ]:
# Generate per-pump baselines with realistic correlations
def generate_pump_baselines(n_pumps, seed=42):
    np.random.seed(seed)
    baselines = []
    for pump_id in range(1, n_pumps + 1):
        baseline = {'pump_id': pump_id}
        
        # Base operating point (affects correlated measures)
        load_factor = np.random.uniform(0.3, 0.9)
        
        for feature, (lo, hi) in NOMINAL_RANGES.items():
            if feature in ['flow_rate', 'motor_current', 'pump_speed']:
                # Correlated with load
                center = lo + (hi - lo) * (0.2 + 0.6 * load_factor)
            elif feature in ['bearing_temp', 'casing_temp']:
                # Higher load = higher temp
                center = lo + (hi - lo) * (0.3 + 0.4 * load_factor)
            else:
                center = lo + (hi - lo) * np.random.uniform(0.2, 0.8)
            baseline[f'{feature}_baseline'] = center
        baselines.append(baseline)
    return pd.DataFrame(baselines)

train_baselines = generate_pump_baselines(TRAIN_PUMPS, seed=42)
print(f"Generated baselines for {len(train_baselines)} pumps")

In [ ]:
# Assign failure schedules - 50% normal, 50% with distributed failures
def assign_training_schedules(n_pumps, n_days, seed=42):
    np.random.seed(seed)
    schedules = []
    n_normal = n_pumps // 2
    
    # Normal pumps
    for i in range(1, n_normal + 1):
        schedules.append({'pump_id': i, 'failure_mode': 'NORMAL', 'onset_day': None, 'failure_day': None, 'window_days': None})
    
    # Failing pumps - distribute across modes
    modes = list(FAILURE_MODES.keys())
    for i in range(n_normal + 1, n_pumps + 1):
        mode = modes[(i - n_normal - 1) % len(modes)]
        window_min, window_max = FAILURE_MODES[mode]
        window = np.random.randint(window_min, window_max + 1)
        
        # Failure happens somewhere in the training window
        failure_day = np.random.randint(window + 5, min(n_days - 2, n_days))
        onset_day = failure_day - window
        
        schedules.append({
            'pump_id': i, 
            'failure_mode': mode, 
            'onset_day': max(1, onset_day), 
            'failure_day': failure_day, 
            'window_days': window
        })
    
    return pd.DataFrame(schedules)

train_schedules = assign_training_schedules(TRAIN_PUMPS, TRAIN_DAYS)
print("Failure distribution:")
print(train_schedules['failure_mode'].value_counts())

In [ ]:
# Failure signatures with GRADUAL + ACCELERATING degradation
def generate_failure_signature(mode, progress, seed_offset=0):
    """
    progress: 0.0 (onset) to 1.0 (failure)
    Returns multiplicative/additive mods per sensor
    Degradation accelerates: uses progress^1.5 for acceleration
    """
    np.random.seed(int(progress * 1000) + seed_offset)
    mods = {k: 0 for k in NOMINAL_RANGES.keys()}
    
    # Accelerating degradation curve
    p = progress ** 1.5
    
    # Add some noise to make it realistic
    noise = 1 + np.random.uniform(-0.1, 0.1)
    
    if mode == 'BEARING_WEAR':
        mods['vibration_rms'] = p * 1.2 * noise      # Strong increase
        mods['bearing_temp'] = p * 50 * noise        # Additive temp rise
        mods['motor_current'] = p * 0.2 * noise      # Slight current increase
        mods['casing_temp'] = p * 15 * noise         # Secondary heat
        
    elif mode == 'CAVITATION':
        mods['suction_pressure'] = -p * 0.5 * noise  # Drops significantly
        mods['vibration_rms'] = p * 0.8 * noise      # Erratic vibration
        mods['flow_rate'] = -p * 0.35 * noise        # Flow drops
        mods['discharge_pressure'] = -p * 0.15 * noise
        
    elif mode == 'VALVE_FAILURE':
        mods['valve_position'] = p * 0.6 * noise if np.random.random() > 0.5 else -p * 0.4 * noise
        mods['flow_rate'] = -p * 0.3 * noise
        mods['discharge_pressure'] = -p * 0.25 * noise
        mods['motor_current'] = p * 0.1 * noise
        
    elif mode == 'OVERHEATING':
        mods['bearing_temp'] = p * 80 * noise        # Sharp temp rise
        mods['casing_temp'] = p * 60 * noise
        mods['motor_current'] = p * 0.25 * noise
        mods['vibration_rms'] = p * 0.3 * noise
        
    elif mode == 'SEAL_LEAK':
        mods['leak_rate'] = p * 3.0 * noise          # Strong leak increase
        mods['discharge_pressure'] = -p * 0.2 * noise
        mods['flow_rate'] = -p * 0.15 * noise
        mods['suction_pressure'] = -p * 0.1 * noise
    
    return mods

print("Failure signatures defined with accelerating degradation")

In [ ]:
# Generate telemetry at 5-minute resolution
def generate_telemetry(baselines_df, schedules_df, n_days, start_date, seed=42):
    np.random.seed(seed)
    total_rows = n_days * ROWS_PER_DAY
    all_data = []
    
    for _, pump in baselines_df.iterrows():
        pump_id = pump['pump_id']
        schedule_match = schedules_df[schedules_df['pump_id'] == pump_id]
        if len(schedule_match) == 0:
            continue
        schedule = schedule_match.iloc[0]
        
        is_failing = schedule['failure_mode'] != 'NORMAL'
        onset_day = schedule['onset_day'] if is_failing else None
        failure_day = schedule['failure_day'] if is_failing else None
        failure_mode = schedule['failure_mode']
        window = schedule['window_days'] if is_failing else None
        
        for row_idx in range(total_rows):
            current_day = row_idx // ROWS_PER_DAY + 1
            day_progress = (row_idx % ROWS_PER_DAY) / ROWS_PER_DAY
            ts = start_date + timedelta(minutes=row_idx * INTERVAL_MINUTES)
            
            # OFFLINE state - after failure
            if is_failing and failure_day and current_day > failure_day:
                days_since_failure = current_day - failure_day
                temp_decay = max(0.5, 1 - days_since_failure * 0.1)  # Gradual cooling
                row = {
                    'pump_id': pump_id, 'ts': ts,
                    'suction_pressure': 0, 'discharge_pressure': 0,
                    'flow_rate': 0, 'motor_current': 0, 'pump_speed': 0,
                    'bearing_temp': pump['bearing_temp_baseline'] * temp_decay,
                    'casing_temp': pump['casing_temp_baseline'] * temp_decay,
                    'vibration_rms': 0, 'valve_position': 0,
                    'leak_rate': pump['leak_rate_baseline'] * 3 if failure_mode == 'SEAL_LEAK' else 0
                }
            # Degrading state
            elif is_failing and onset_day and current_day >= onset_day and current_day <= failure_day:
                # Progress within degradation window (0 to 1)
                progress = min((current_day - onset_day + day_progress) / max(window, 1), 1.0)
                mods = generate_failure_signature(failure_mode, progress, seed_offset=pump_id)
                
                row = {'pump_id': pump_id, 'ts': ts}
                for feature in NOMINAL_RANGES.keys():
                    baseline = pump[f'{feature}_baseline']
                    mod = mods.get(feature, 0)
                    noise = np.random.uniform(0.97, 1.03)
                    
                    if feature in ['bearing_temp', 'casing_temp']:
                        # Additive for temperatures
                        value = baseline + mod + np.random.normal(0, 1)
                    else:
                        # Multiplicative for others
                        value = baseline * (1 + mod) * noise
                    row[feature] = max(0, value)
            # Normal operation
            else:
                row = {'pump_id': pump_id, 'ts': ts}
                for feature in NOMINAL_RANGES.keys():
                    baseline = pump[f'{feature}_baseline']
                    noise = np.random.uniform(0.98, 1.02)
                    row[feature] = baseline * noise + np.random.normal(0, baseline * 0.005)
            
            all_data.append(row)
    
    return pd.DataFrame(all_data)

print("Telemetry generator defined")

In [ ]:
# Generate training telemetry
train_start = datetime(2026, 1, 1)
print(f"Generating training telemetry ({TRAIN_PUMPS} pumps × {TRAIN_DAYS} days)...")
print("This may take a few minutes...")
train_telemetry = generate_telemetry(train_baselines, train_schedules, TRAIN_DAYS, train_start, seed=42)
print(f"Generated {len(train_telemetry):,} rows")

In [ ]:
# Feature engineering at 5-minute resolution
def engineer_features(telemetry_df, baselines_df):
    df = telemetry_df.copy().sort_values(['pump_id', 'ts'])
    
    # Derived features
    df['pressure_diff'] = df['discharge_pressure'] - df['suction_pressure']
    df['flow_per_speed'] = df['flow_rate'] / df['pump_speed'].replace(0, 1)
    df['current_per_flow'] = df['motor_current'] / df['flow_rate'].replace(0, 1)
    
    # Rolling windows: 1h=12, 6h=72, 24h=288 intervals
    windows = {'1h': 12, '6h': 72, '24h': 288}
    roll_features = ['vibration_rms', 'bearing_temp', 'casing_temp', 'leak_rate', 
                     'flow_rate', 'suction_pressure', 'valve_position', 'motor_current']
    
    print("Computing rolling features per pump...")
    for pump_id in df['pump_id'].unique():
        mask = df['pump_id'] == pump_id
        pump_df = df.loc[mask].copy()
        
        for feature in roll_features:
            for window_name, window_size in windows.items():
                col_mean = f'{feature}_mean_{window_name}'
                col_std = f'{feature}_std_{window_name}'
                df.loc[mask, col_mean] = pump_df[feature].rolling(window_size, min_periods=1).mean()
                df.loc[mask, col_std] = pump_df[feature].rolling(window_size, min_periods=1).std().fillna(0)
        
        # Slopes (24h trend)
        for feature in ['vibration_rms', 'bearing_temp', 'casing_temp', 'leak_rate', 'flow_rate']:
            rolling = pump_df[feature].rolling(288, min_periods=12)
            slope = rolling.apply(lambda x: (x.iloc[-1] - x.iloc[0]) / max(len(x), 1) if len(x) > 1 else 0, raw=False)
            df.loc[mask, f'{feature}_slope_24h'] = slope
    
    # Merge baselines for z-score computation
    df = df.merge(baselines_df, on='pump_id', how='left')
    
    # Z-scores (deviation from baseline)
    for feature in ['vibration_rms', 'bearing_temp', 'leak_rate', 'flow_rate']:
        baseline_col = f'{feature}_baseline'
        if baseline_col in df.columns:
            df[f'{feature}_zscore'] = (df[feature] - df[baseline_col]) / (df[baseline_col] * 0.1 + 0.001)
    
    return df

print("Engineering features...")
train_features = engineer_features(train_telemetry, train_baselines)
print(f"Features: {len(train_features.columns)} columns")

In [ ]:
# Create labels - early 25% = NORMAL, rest = failure mode
def create_labels(features_df, schedules_df, start_date):
    df = features_df.copy()
    df['label'] = 'NORMAL'
    df['rul_days'] = None
    
    for _, schedule in schedules_df.iterrows():
        pump_id = schedule['pump_id']
        mode = schedule['failure_mode']
        if mode == 'NORMAL':
            continue
        
        onset_day = schedule['onset_day']
        failure_day = schedule['failure_day']
        window = failure_day - onset_day
        
        # Detection starts at 25% into degradation
        detection_day = onset_day + int(window * 0.25)
        
        mask = df['pump_id'] == pump_id
        for idx in df[mask].index:
            ts = df.loc[idx, 'ts']
            current_day = (ts - start_date).days + 1
            day_fraction = (ts - start_date).seconds / 86400
            
            if current_day > failure_day:
                df.loc[idx, 'label'] = 'OFFLINE'
                df.loc[idx, 'rul_days'] = None
            elif current_day >= detection_day:
                df.loc[idx, 'label'] = mode
                # RUL with slight variance (not perfect countdown)
                base_rul = failure_day - current_day - day_fraction
                variance = np.random.uniform(-0.1, 0.1)
                df.loc[idx, 'rul_days'] = max(0, base_rul + variance)
    
    return df

print("Creating labels...")
train_labeled = create_labels(train_features, train_schedules, train_start)
print("Label distribution:")
print(train_labeled['label'].value_counts())

In [ ]:
# Define feature columns for training
FEATURE_COLS = [
    # Raw measures
    'suction_pressure', 'discharge_pressure', 'flow_rate', 'motor_current',
    'pump_speed', 'bearing_temp', 'casing_temp', 'vibration_rms', 'valve_position', 'leak_rate',
    # Derived
    'pressure_diff', 'flow_per_speed', 'current_per_flow',
    # Rolling means
    'vibration_rms_mean_1h', 'vibration_rms_mean_6h', 'vibration_rms_mean_24h',
    'bearing_temp_mean_1h', 'bearing_temp_mean_6h', 'bearing_temp_mean_24h',
    'leak_rate_mean_1h', 'leak_rate_mean_6h', 'leak_rate_mean_24h',
    'flow_rate_mean_1h', 'flow_rate_mean_6h', 'flow_rate_mean_24h',
    'motor_current_mean_1h', 'motor_current_mean_24h',
    # Rolling stds
    'vibration_rms_std_6h', 'flow_rate_std_6h', 'suction_pressure_std_6h',
    'bearing_temp_std_6h', 'leak_rate_std_6h',
    # Slopes
    'vibration_rms_slope_24h', 'bearing_temp_slope_24h', 'leak_rate_slope_24h', 
    'flow_rate_slope_24h', 'casing_temp_slope_24h',
    # Z-scores
    'vibration_rms_zscore', 'bearing_temp_zscore', 'leak_rate_zscore', 'flow_rate_zscore'
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_labeled.columns]
print(f"Using {len(FEATURE_COLS)} features for training")

In [ ]:
# Train XGBoost Classifier
print("Training classifier...")
X = train_labeled[FEATURE_COLS].fillna(0)
y = train_labeled['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_labels = list(le.classes_)
print(f"Classes (alphabetical order): {class_labels}")

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

classifier = XGBClassifier(
    n_estimators=150, 
    max_depth=8, 
    learning_rate=0.1, 
    random_state=42, 
    eval_metric='mlogloss',
    n_jobs=-1
)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nClassifier Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_labels))

In [ ]:
# Train RUL Regressors (one per failure mode)
rul_models = {}

for mode in FAILURE_MODES.keys():
    print(f"\nTraining RUL model for {mode}...")
    mode_data = train_labeled[(train_labeled['label'] == mode) & (train_labeled['rul_days'].notna())]
    
    if len(mode_data) < 500:
        print(f"  Skipping - only {len(mode_data)} samples (need 500+)")
        continue
    
    X_rul = mode_data[FEATURE_COLS].fillna(0)
    y_rul = mode_data['rul_days'].astype(float)
    X_train_rul, X_test_rul, y_train_rul, y_test_rul = train_test_split(X_rul, y_rul, test_size=0.2, random_state=42)
    
    model = XGBRegressor(
        n_estimators=150, 
        max_depth=6, 
        learning_rate=0.1, 
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_rul, y_train_rul)
    
    y_pred_rul = model.predict(X_test_rul)
    mae = mean_absolute_error(y_test_rul, y_pred_rul)
    r2 = r2_score(y_test_rul, y_pred_rul)
    
    print(f"  Samples: {len(mode_data):,}, MAE: {mae:.2f} days, R²: {r2:.3f}")
    rul_models[mode] = model

print(f"\nTrained {len(rul_models)} RUL models")

In [ ]:
# Store class labels for inference
CLASS_LABELS = class_labels
print(f"Class labels to use at inference: {CLASS_LABELS}")

# Register models to Snowflake Model Registry
registry = Registry(session=session, database_name='PDM_DEMO', schema_name='ML')

# Register classifier
print("\nRegistering PDM_CLASSIFIER...")
sample_input = X_train.head(10)
try:
    registry.delete_model('PDM_CLASSIFIER')
except:
    pass

registry.log_model(
    model=classifier,
    model_name='PDM_CLASSIFIER',
    version_name='V3',
    sample_input_data=sample_input,
    comment=f'7-class pump failure classifier. Labels: {CLASS_LABELS}'
)
print("  Registered PDM_CLASSIFIER V3")

In [ ]:
# Register RUL models
for mode, model in rul_models.items():
    model_name = f'PDM_RUL_{mode}'
    print(f"Registering {model_name}...")
    try:
        registry.delete_model(model_name)
    except:
        pass
    
    registry.log_model(
        model=model,
        model_name=model_name,
        version_name='V3',
        sample_input_data=sample_input,
        comment=f'RUL regressor for {mode}'
    )
    print(f"  Registered {model_name} V3")

print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETE!")
print("="*60)
print(f"\nClass labels (SAVE THIS): {CLASS_LABELS}")
print("\nModels registered:")
print("  - PDM_CLASSIFIER V3")
for mode in rul_models.keys():
    print(f"  - PDM_RUL_{mode} V3")